In [30]:
import re

In [1]:
'''
Tren Meckel, Jack Rollo
October 2nd, 2024
This program parses a Python source file to identify and report on function calls and imports.
It extracts:
- Imported modules and their shorten name.
- User-defined functions.
- Function calls, including object methods.

The output includes the function names, types (built-in, module, object, user-defined), and their parameters.
Call the `main()` function with the filename to generate the report.

'''


def read_file(filename):
    '''
    This function reads in the program file.
    
    Parameters:
        filename (str): The file to be read.
    
    Return Value:
        str: The content of the file as a string.
    '''
    with open(filename, 'r') as file:
        return file.read()

def find_imports(code):
    '''
    This function finds any import statements in the file and records them.
    
    Parameters:
        code (str): The source code to parse.
    
    Return Value:
        dict: A dictionary where the keys are the imported modules (or their shortened names, if using 'as')
              and the values are the original module names.
              Example: {'random': 'random', 'ran': 'random'}
    '''
    
    imports = {}  # Dictionary to store imports
    import_pattern = r'import (\w+)(?: as (\w+))?'
    import_matches = re.findall(import_pattern, code)

    # Loop through all matches to store them in dictionary
    for match in import_matches:
        module_name = match[0] # og module name
        import_as = match[1] if match[1] else module_name  # Use shorten name if present, otherwise module name
        imports[import_as] = module_name # Store in dictionary with the shorten name as key
        
    return imports

def find_user_functions(code):
    '''
    This function finds any user-defined functions in the code.
    
    Parameters:
        code (str): The source code to parse.
    
    Return Value:
        list: A list of user-defined function names.
    '''
    
    def_pattern = r'def\s+(\w+)\s*\('   
    user_functions = re.findall(def_pattern, code)
    return user_functions

def find_function_calls(code):
    """
    This function finds all function calls in the given code and distinguishes
    between object methods and regular function calls.
    
    Parameters:
        code (str) : The source code to parse.
    
    Return Value:
        A tuple containing two lists:
        - A list of object methods (name, method, parameters)
        - A list of regular function calls (name, parameters)
    """
    
    # Pattern to find object method calls (like 'object.method(params)')
    object_methods = re.findall(r'(\w+)\.(\w+)\(([^)]*)\)', code) 
    
    # Pattern to find regular function calls (like 'function(params)')
    function_calls = re.findall(r'(\w+)\(([^)]*)\)', code)  
    
    
    # List to hold unique object method names for reporting
    object_method_names = []
    for object_name, method_name, parameters in object_methods: # Iterate through each object method found in the code
        # Check if the method name is not already in the list of unique method names
        if method_name not in object_method_names:
            object_method_names.append(method_name)  # Add unique object method names
    
    # Filter regular function calls by excluding object methods
    regular_function_calls = []
    for function_name, parameters in function_calls: # Iterate through each function call found in the code
        # Check if the function name is not in the list of object method names
        if function_name not in object_method_names:
            regular_function_calls.append((function_name, parameters))  # Only include non-object methods
    
    return object_methods, regular_function_calls

def generate_report(imports, user_functions, object_methods, function_calls):
    '''
    This function generates a report detailing the types of function calls and their parameters.
    
    Parameters:
        imports (dict): A dictionary of imported modules.
        user_functions (list): A list of user-defined functions.
        object_methods (list of tuples): A list of object method calls with (object, method, parameters).
        function_calls (list of tuples): A list of regular function calls with (function_name, parameters).
    
    Return Value:
        None. Prints the report directly.
    '''
    
    # --- Report for object method calls (e.g., obj.method()) ---
    for object, function_name, parameters in object_methods: 
        if object in imports:
            # if the object is actually a module, put it under module
            print("Function Name:   ", function_name)
            print("Function Type:   module")
            print("Module:          ", imports[object])
        else:
            # Otherwise, it is truly an object method
            print("Function Name:   ", function_name)
            print("Function Type:   object")
            print("Object:          ", object)

        # List and print the parameters of the method
        print("Parameters: ")
        parameter_list = [parameter.strip() for parameter in parameters.split(',')] 
        for parameter in parameter_list:
            print( "\t" , parameter )
        print() # Empty line for better readability
    
    
    # --- Report for regular function calls (e.g., function()) ---
    for function_name, parameters in function_calls:
        if function_name in user_functions:
            # If the function name matches a user-defined function
            print("Function Name:   ", function_name)
            print("Function Type:   user-defined")
        elif function_name in imports:
            print("Function Name:   ", function_name)
            print("Function Type:   module")
            print("Module:          ", imports[function_name])
        else:
            # Otherwise, it's considered a built-in function
            print("Function Name:   ", function_name)
            print("Function Type:   built-in")

        # List and print the parameters of the method
        print( "Parameters: " ) 
        parameter_list = [parameter.strip() for parameter in parameters.split(',')] 
        for parameter in parameter_list:
            print( "\t" , parameter )
        print() # Empty line for better readability

def main(filename):
    '''
    The main function that coordinates the reading, parsing, and reporting of function calls in the source file.
    
    Parameters:
        filename (str): The name of the file to be read
    
    Return Value:
        None. Generates a report by calling other functions.
    '''
       
    # 1) read the file
    code = read_file(filename)
    
    # 2) Parse code to find imports, user-defined functions, objectmethods and function calls
    imports = find_imports(code)
    user_functions = find_user_functions(code)
    object_methods, function_calls = find_function_calls(code)

    # Give report based on what was found
    generate_report(imports, user_functions, object_methods, function_calls)


In [26]:
# Example to follow
main('sampleProg.py')

Function Name:    randint
Function Type:   module
Module:           random
Parameters: 
	 0
	 10

Function Name:    append
Function Type:   object
Object:           nums
Parameters: 
	 value

Function Name:    range
Function Type:   built-in
Parameters: 
	 10

Function Name:    print
Function Type:   built-in
Parameters: 
	 'numbers:'
	 nums

Function Name:    findLargest
Function Type:   user-defined
Parameters: 
	 nums

Function Name:    print
Function Type:   built-in
Parameters: 
	 'largest:'
	 largest

Function Name:    main
Function Type:   user-defined
Parameters: 
	 



In [24]:
# Our test file
main('test.py')

Function Name:    sqrt
Function Type:   module
Module:           math
Parameters: 
	 16

Function Name:    randint
Function Type:   module
Module:           random
Parameters: 
	 1
	 10

Function Name:    sort
Function Type:   object
Object:           test_list
Parameters: 
	 

Function Name:    upper
Function Type:   object
Object:           name
Parameters: 
	 

Function Name:    append
Function Type:   object
Object:           test_list
Parameters: 
	 42

Function Name:    findLargest
Function Type:   user-defined
Parameters: 
	 numbers

Function Name:    max
Function Type:   built-in
Parameters: 
	 numbers

Function Name:    greet
Function Type:   user-defined
Parameters: 
	 name

Function Name:    print
Function Type:   built-in
Parameters: 
	 "Hello"
	 {name}"!"

Function Name:    add_numbers
Function Type:   user-defined
Parameters: 
	 a
	 b

Function Name:    square
Function Type:   user-defined
Parameters: 
	 x

Function Name:    print
Function Type:   built-in
Parameters: 
	 